# Lab 0: Setup-Check

Dieses Notebook prüft, ob Ihre Umgebung für den Seminartag bereit ist: Python-Version, die Pakete `crewai`, `crewai-tools`, `mcp` und `openai`, das `crewai`-Kommandozeilenwerkzeug, die Verbindung zum LLM-Endpunkt, Git und (optional) der GitHub-Token für Lab 4. Jede Zelle druckt eine Zeile mit `OK` oder `FEHLT` und sagt, was zu tun ist. Es gibt hier keine Aufgaben.

Vorher: `labs/.env.example` nach `labs/.env` kopieren und die Werte an Ihren LLM-Endpunkt anpassen (LM Studio ist der Default).

Erwartete Ergebnisse stehen in EXPECTED_RESULTS.md.

In [ ]:
from pathlib import Path
# Setup: Umgebungsvariablen laden, Python-Version prüfen
import sys

import os
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env" if (Path.cwd() / ".env").exists() else Path.cwd() / "labs" / ".env", override=True)  # labs/.env gilt vor jeder anderen .env
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:1234/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "lm-studio")
LLM_MODEL = os.environ.get("LLM_MODEL", "qwen/qwen3.6-35b-a3b")

ERGEBNISSE = {}

def melde(name: str, ok: bool, info: str = "", hinweis: str = "") -> None:
    """Druckt eine OK/FEHLT-Zeile und merkt sich das Ergebnis für die Zusammenfassung."""
    ERGEBNISSE[name] = ok
    status = "OK    " if ok else "FEHLT "
    zeile = f"[{status}] {name}"
    if info:
        zeile += f": {info}"
    if not ok and hinweis:
        zeile += f"\n         -> {hinweis}"
    print(zeile)

v = sys.version_info
melde("Python-Version", (3, 10) <= (v.major, v.minor) < (3, 14),
      f"{v.major}.{v.minor}.{v.micro} ({sys.executable})",
      "CrewAI verlangt Python >= 3.10 und < 3.14. Umgebung mit `uv venv --python 3.12` neu anlegen.")
print(f"LLM-Endpunkt laut Konfiguration: {LLM_BASE_URL}, Modell: {LLM_MODEL}")

In [ ]:
# Pakete importieren und Versionen zeigen
from importlib import import_module
from importlib.metadata import version, PackageNotFoundError

PAKETE = [  # (Importname, Distributionsname, Installationshinweis)
    ("crewai", "crewai", "uv add crewai"),
    ("crewai_tools", "crewai-tools", "uv add crewai-tools"),
    ("mcp", "mcp", "uv add mcp"),
    ("openai", "openai", "uv add openai"),
]
for modul, dist, hinweis in PAKETE:
    try:
        import_module(modul)
        try:
            ver = version(dist)
        except PackageNotFoundError:
            ver = "Version unbekannt"
        melde(f"Paket {dist}", True, ver)
    except ImportError as e:
        melde(f"Paket {dist}", False, str(e), f"Installieren mit `{hinweis}`, danach Kernel neu starten.")

In [ ]:
# crewai-Kommandozeile (wird für `crewai create crew ...` und `crewai run` gebraucht)
import shutil
import subprocess
from pathlib import Path

# Erst im PATH suchen, dann neben dem aktuellen Python-Interpreter (venv/bin).
cli = shutil.which("crewai") or str(Path(sys.executable).parent / "crewai")
try:
    proc = subprocess.run([cli, "version"], capture_output=True, text=True, timeout=60)
    ausgabe = (proc.stdout + proc.stderr).strip()
    melde("crewai CLI", proc.returncode == 0, f"{ausgabe} ({cli})",
          "Ausgabe prüfen; bei Bedarf `uv tool install crewai` oder `uv add crewai` erneut ausführen.")
except FileNotFoundError:
    melde("crewai CLI", False, "Kommando nicht gefunden",
          "`uv tool install crewai` ausführen oder das venv aktivieren, in dem crewai installiert ist.")
except subprocess.TimeoutExpired:
    melde("crewai CLI", False, "keine Antwort in 60 s", "Aufruf im Terminal wiederholen: `crewai version`.")

In [ ]:
# Verbindung zum LLM-Endpunkt: Modellliste und ein Ein-Wort-Aufruf mit Laufzeit
import time
from openai import OpenAI

client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
try:
    modelle = [m.id for m in client.models.list().data]
    melde("LLM-Endpunkt erreichbar", True, f"{len(modelle)} Modelle unter {LLM_BASE_URL}")
    melde("Modell verfügbar", LLM_MODEL in modelle, LLM_MODEL,
          "Modellname in .env prüfen (LLM_MODEL) oder Modell in LM Studio/Ollama laden. Gefunden: "
          + ", ".join(modelle[:8]))
    t0 = time.perf_counter()
    antwort = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "Antworte mit genau einem Wort: OK"}],
        temperature=0.0,
    )
    dauer = time.perf_counter() - t0
    text = (antwort.choices[0].message.content or "").strip()
    melde("Chat-Aufruf", bool(text), f"Antwort {text!r} in {dauer:.1f} s, {antwort.usage.total_tokens} Tokens",
          "Leere Antwort: Modell in LM Studio laden oder Kontextlänge prüfen.")
    if dauer > 20:
        print("         Hinweis: über 20 s für einen Ein-Wort-Aufruf. Kleineres Modell wählen oder GPU-Offload prüfen.")
except Exception as e:
    melde("LLM-Endpunkt erreichbar", False, f"{type(e).__name__}: {e}",
          "LM Studio starten (Server-Tab, Port 1234) bzw. `ollama serve`; LLM_BASE_URL und LLM_API_KEY in .env prüfen.")
    ERGEBNISSE.setdefault("Modell verfügbar", False)
    ERGEBNISSE.setdefault("Chat-Aufruf", False)

In [ ]:
# Git und GitHub-Konfiguration (kein Netzaufruf; der Token wird nur auf Vorhandensein geprüft)
try:
    proc = subprocess.run(["git", "--version"], capture_output=True, text=True, timeout=30)
    melde("git", proc.returncode == 0, proc.stdout.strip(), "Git installieren: https://git-scm.com/downloads")
except FileNotFoundError:
    melde("git", False, "Kommando nicht gefunden", "Git installieren: https://git-scm.com/downloads")

token = os.environ.get("GITHUB_TOKEN", "")
repo = os.environ.get("GITHUB_REPO", "")
if token:
    print(f"[OK    ] GITHUB_TOKEN gesetzt ({len(token)} Zeichen, Wert wird nicht angezeigt)")
else:
    print("[INFO  ] GITHUB_TOKEN nicht gesetzt: Lab 4 läuft dann offline gegen das lokale Demo-Repo. Optional in .env eintragen.")
print(f"[INFO  ] GITHUB_REPO = {repo or '(leer)'}")

In [ ]:
# Zusammenfassung
pflicht = [n for n in ERGEBNISSE]  # alle geprüften Punkte sind Pflicht; GitHub ist optional und steht nicht in ERGEBNISSE
fehlend = [n for n, ok in ERGEBNISSE.items() if not ok]
print("=" * 60)
for name, ok in ERGEBNISSE.items():
    print(f"  {'OK   ' if ok else 'FEHLT'}  {name}")
print("=" * 60)
if fehlend:
    print(f"{len(fehlend)} von {len(pflicht)} Prüfungen offen: {', '.join(fehlend)}. Hinweise oben beachten.")
else:
    print(f"Alle {len(pflicht)} Prüfungen bestanden. Die Umgebung ist bereit für Lab 1.")

## Weiter

Wenn alle Prüfungen `OK` melden, öffnen Sie `lab_1_react_loop.ipynb`. Dort bauen Sie die Agentenschleife von Hand gegen genau diesen LLM-Endpunkt.